# Table of Contents
## 1. Load csv files into dataframes
## 2. Drops rows from 'secs' dataframe
## 3. Merges dataframes and assigns merged dataframe to 'barsCombined' and saves to 'newBarsCombined.csv'
## 4. Renames columns in 'barCombined'
## 5. Parses 'date' column and creates 'day', 'month', 'year', and 'confirm_time' columns.
## 6. Ensures column titles are lowercase
## 7. Fills NAs
## 8. Ensure time is a string
## 9. Creates 'day_of_week' column (Mon-Fri)
## 10. Drops specific dates from 'barsCombined'
## 11. Creates 'market_open_09:30' column to show opening price of market for each days rows
## 12. Creates 'initial_direction' row which indicates the movement of the market for that row/time in relation to 'market_open_09:30' and saves it to 'newBarsCombined.csv'
## 13. Merges dataframes created 'cleaning' dataframe that now has 'contract_location' column and saves to 'cleaning.csv'

## 1.

In [8]:
import pandas as pd

# Load the CSV files/Datasets
barsCombined = pd.read_csv("GoodOldGoodOld.csv")
secs = pd.read_csv("NewGoodOldGoodOld.csv")

# Inspect the first few rows
print(barsCombined.info())
print(secs.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 28935 entries, 0 to 28934
Data columns (total 9 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   date      28935 non-null  object 
 1   open      28935 non-null  float64
 2   high      28935 non-null  float64
 3   low       28935 non-null  float64
 4   close     28935 non-null  float64
 5   volume    28935 non-null  float64
 6   average   28935 non-null  float64
 7   barCount  28935 non-null  int64  
 8   time      28935 non-null  object 
dtypes: float64(6), int64(1), object(2)
memory usage: 2.0+ MB
None
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10621 entries, 0 to 10620
Data columns (total 9 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   date      10621 non-null  object 
 1   open      10621 non-null  float64
 2   high      10621 non-null  float64
 3   low       10621 non-null  float64
 4   close     10621 non-null  float64
 5   volume    

## 2.
## These rows had no volume

In [11]:
# Drop rows with zero volume (pre-market data before an identified gap)
print(secs.head())

# NOTE: The number 1140 represents the exact count of zero-volume rows
# from the IBKR 1-sec feed before a known data gap in NewGoodOldGoodOld.csv.
# If new sec data is ever prepended to that file, this number must be
# recalculated. A more robust alternative that auto-detects zero-volume rows:
#   secs = secs[secs['volume'] > 0].reset_index(drop=True)
secs = secs.iloc[1140:].reset_index(drop=True)

print(secs.head())


                        date     open     high      low    close  volume  \
0  2022-12-12 09:30:01-05:00  4020.25  4020.25  4020.25  4020.25     0.0   
1  2022-12-12 09:30:02-05:00  4020.25  4020.25  4020.25  4020.25     0.0   
2  2022-12-12 09:30:03-05:00  4020.25  4020.25  4020.25  4020.25     0.0   
3  2022-12-12 09:30:05-05:00  4020.25  4020.25  4020.25  4020.25     0.0   
4  2022-12-12 09:30:08-05:00  4020.25  4020.25  4020.25  4020.25     0.0   

   average  barCount      time  
0  4020.25         0  09:30:01  
1  4020.25         0  09:30:02  
2  4020.25         0  09:30:03  
3  4020.25         0  09:30:05  
4  4020.25         0  09:30:08  
                        date     open     high      low    close  volume  \
0  2023-03-10 09:30:01-05:00  3949.00  3949.50  3948.50  3949.25   327.0   
1  2023-03-10 09:30:02-05:00  3949.25  3950.25  3949.25  3950.00   160.0   
2  2023-03-10 09:30:03-05:00  3950.00  3950.00  3949.25  3949.75   127.0   
3  2023-03-10 09:30:05-05:00  3949.50  39

## 3.

In [14]:
# Merge the datasets on ['date', 'candle_open_price', 'candle_high_price', 'candle_low_price', 
#'candle_close_price', 'volume', 'average', 'bar_count', 'time',
#'confirm_time', 'year', 'month', 'day', 'day_of_week']

barsCombined = pd.merge(
    barsCombined,
    secs,
    on=['date', 'open', 'high', 'low', 'close', 'volume', 'average', 'barCount',
       'time'],  # Align on these keys
    how="outer"  # Include all rows from both datasets
)

# Sort the final dataset by 'id' in ascending order
barsCombined = barsCombined.sort_values(by=["date", 'time'], ascending=True).reset_index(drop=True)

# Save the final sorted dataset to a CSV file
barsCombined.to_csv("newBarsCombined.csv", index=False)
print("barsCombined dataset sorted and saved as 'newBarsCombined.csv'")

barsCombined dataset sorted and saved as 'newBarsCombined.csv'


## 4.

In [17]:
barsCombined= barsCombined.rename(columns = {'open':'candle_open_price', 'high': 'candle_high_price', 'low':'candle_low_price',
                                     'close':'candle_close_price', 'average_price':'candle_average_price', 'barCount':'bar_count'})
barsCombined.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 38416 entries, 0 to 38415
Data columns (total 9 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   date                38416 non-null  object 
 1   candle_open_price   38416 non-null  float64
 2   candle_high_price   38416 non-null  float64
 3   candle_low_price    38416 non-null  float64
 4   candle_close_price  38416 non-null  float64
 5   volume              38416 non-null  float64
 6   average             38416 non-null  float64
 7   bar_count           38416 non-null  int64  
 8   time                38416 non-null  object 
dtypes: float64(6), int64(1), object(2)
memory usage: 2.6+ MB


## 5.

In [20]:
import pandas as pd
from datetime import datetime

# Check the first few values
print(barsCombined["date"].head())

# Step 1: Remove the timezone information while keeping the local time
try:
    barsCombined["date"] = barsCombined["date"].apply(lambda x: datetime.strptime(x[:-6], "%Y-%m-%d %H:%M:%S") if isinstance(x, str) else x)
    print("✅ Date column successfully parsed without timezone shift!\n")
except Exception as e:
    print(f"❌ Error while parsing date column: {e}")

# Step 2: Ensure 'date' is converted properly
if not pd.api.types.is_datetime64_any_dtype(barsCombined["date"]):
    print("⚠ Warning: 'date' column is not in datetime format. Check your data!")
else:
    print("✅ 'date' column is confirmed as datetime!\n")

# Step 3: Extract time and separate components
if "date" in barsCombined.columns and pd.api.types.is_datetime64_any_dtype(barsCombined["date"]):
    barsCombined["confirm_time"] = barsCombined["date"].dt.strftime("%H:%M:%S")  # Extract local time
    barsCombined["date"] = barsCombined["date"].dt.date  # Keep only the date (YYYY-MM-DD)
    print("✅ Successfully extracted 'confirm_time' and cleaned 'date'.\n")
else:
    print("⚠ Skipping extraction because 'date' is not in datetime format.")

# Step 4: Extract Year, Month, and Day into separate columns
if "date" in barsCombined.columns:
    barsCombined["year"] = pd.to_datetime(barsCombined["date"], errors="coerce").dt.year
    barsCombined["month"] = pd.to_datetime(barsCombined["date"], errors="coerce").dt.month
    barsCombined["day"] = pd.to_datetime(barsCombined["date"], errors="coerce").dt.day
    print("✅ 'year', 'month', and 'day' columns created successfully!\n")
else:
    print("⚠ Skipping year, month, and day extraction as 'date' column does not exist.")

# Step 5: Display the updated dataset
print(barsCombined[["date", "confirm_time", "year", "month", "day"]].head())


0    2022-12-12 09:30:00-05:00
1    2022-12-12 09:31:00-05:00
2    2022-12-12 09:32:00-05:00
3    2022-12-12 09:33:00-05:00
4    2022-12-12 09:34:00-05:00
Name: date, dtype: object
✅ Date column successfully parsed without timezone shift!

✅ 'date' column is confirmed as datetime!

✅ Successfully extracted 'confirm_time' and cleaned 'date'.

✅ 'year', 'month', and 'day' columns created successfully!

         date confirm_time  year  month  day
0  2022-12-12     09:30:00  2022     12   12
1  2022-12-12     09:31:00  2022     12   12
2  2022-12-12     09:32:00  2022     12   12
3  2022-12-12     09:33:00  2022     12   12
4  2022-12-12     09:34:00  2022     12   12


## 6.

In [23]:
# Import necessary libraries
import pandas as pd
from datetime import datetime

# Ensure all column names are lowercase for consistency
barsCombined.columns = barsCombined.columns.str.lower()

print(barsCombined.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 38416 entries, 0 to 38415
Data columns (total 13 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   date                38416 non-null  object 
 1   candle_open_price   38416 non-null  float64
 2   candle_high_price   38416 non-null  float64
 3   candle_low_price    38416 non-null  float64
 4   candle_close_price  38416 non-null  float64
 5   volume              38416 non-null  float64
 6   average             38416 non-null  float64
 7   bar_count           38416 non-null  int64  
 8   time                38416 non-null  object 
 9   confirm_time        38416 non-null  object 
 10  year                38416 non-null  int32  
 11  month               38416 non-null  int32  
 12  day                 38416 non-null  int32  
dtypes: float64(6), int32(3), int64(1), object(3)
memory usage: 3.4+ MB
None


## 7.

In [26]:
if barsCombined[['year', 'month', 'day']].isna().any().any():
    barsCombined["year"] = barsCombined["year"].fillna(0).astype(int)
    barsCombined["month"] = barsCombined["month"].fillna(0).astype(int)
    barsCombined["day"] = barsCombined["day"].fillna(0).astype(int)
    print("NA values existed in nadex and were filled.")
else:
    print("No NA values were found in barsCombined 'year', 'month', 'day' columns!")

print(barsCombined.head())

No NA values were found in barsCombined 'year', 'month', 'day' columns!
         date  candle_open_price  candle_high_price  candle_low_price  \
0  2022-12-12            3976.25            3981.50           3974.75   
1  2022-12-12            3980.00            3980.50           3976.00   
2  2022-12-12            3977.25            3979.25           3975.25   
3  2022-12-12            3978.50            3979.75           3975.50   
4  2022-12-12            3978.50            3979.00           3973.75   

   candle_close_price   volume   average  bar_count      time confirm_time  \
0             3980.00  13850.0  3977.425       4176  09:30:00     09:30:00   
1             3977.50   8373.0  3977.900       2593  09:31:00     09:31:00   
2             3978.50   5649.0  3977.150       1895  09:32:00     09:32:00   
3             3978.75   5903.0  3978.200       2027  09:33:00     09:33:00   
4             3974.50   5902.0  3975.675       1996  09:34:00     09:34:00   

   year  month  day 

## 8.

In [29]:
# Ensure 'time' is a string in nadex

barsCombined["time"] = barsCombined["time"].astype(str)

barsCombined["year"] = barsCombined["year"].astype(int)
barsCombined["month"] = barsCombined["month"].astype(int)
barsCombined["day"] = barsCombined["day"].astype(int)

print(barsCombined.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 38416 entries, 0 to 38415
Data columns (total 13 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   date                38416 non-null  object 
 1   candle_open_price   38416 non-null  float64
 2   candle_high_price   38416 non-null  float64
 3   candle_low_price    38416 non-null  float64
 4   candle_close_price  38416 non-null  float64
 5   volume              38416 non-null  float64
 6   average             38416 non-null  float64
 7   bar_count           38416 non-null  int64  
 8   time                38416 non-null  object 
 9   confirm_time        38416 non-null  object 
 10  year                38416 non-null  int32  
 11  month               38416 non-null  int32  
 12  day                 38416 non-null  int32  
dtypes: float64(6), int32(3), int64(1), object(3)
memory usage: 3.4+ MB
None


In [31]:
barsCombined.head()

,date,candle_open_price,candle_high_price,candle_low_price,candle_close_price,volume,average,bar_count,time,confirm_time,year,month,day
0,2022-12-12,3976.25,3981.50,3974.75,3980.00,13850.0,3977.425,4176,09:30:00,09:30:00,2022,12,12
1,2022-12-12,3980.00,3980.50,3976.00,3977.50,8373.0,3977.900,2593,09:31:00,09:31:00,2022,12,12
2,2022-12-12,3977.25,3979.25,3975.25,3978.50,5649.0,3977.150,1895,09:32:00,09:32:00,2022,12,12
3,2022-12-12,3978.50,3979.75,3975.50,3978.75,5903.0,3978.200,2027,09:33:00,09:33:00,2022,12,12
4,2022-12-12,3978.50,3979.00,3973.75,3974.50,5902.0,3975.675,1996,09:34:00,09:34:00,2022,12,12


## 9.

In [34]:
import pandas as pd

# Vectorized day-of-week using pandas datetime accessor
# Replaces a slow row-by-row .apply() lambda
barsCombined["day_of_week"] = pd.to_datetime(barsCombined["date"]).dt.day_name()

barsCombined.head(15850)


,date,candle_open_price,candle_high_price,candle_low_price,candle_close_price,volume,average,bar_count,time,confirm_time,year,month,day,day_of_week
0,2022-12-12,3976.25,3981.50,3974.75,3980.00,13850.0,3977.425,4176,09:30:00,09:30:00,2022,12,12,Monday
1,2022-12-12,3980.00,3980.50,3976.00,3977.50,8373.0,3977.900,2593,09:31:00,09:31:00,2022,12,12,Monday
2,2022-12-12,3977.25,3979.25,3975.25,3978.50,5649.0,3977.150,1895,09:32:00,09:32:00,2022,12,12,Monday
3,2022-12-12,3978.50,3979.75,3975.50,3978.75,5903.0,3978.200,2027,09:33:00,09:33:00,2022,12,12,Monday
4,2022-12-12,3978.50,3979.00,3973.75,3974.50,5902.0,3975.675,1996,09:34:00,09:34:00,2022,12,12,Monday
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15845,2023-11-24,4567.00,4567.00,4566.25,4567.00,1850.0,4566.850,544,09:55:00,09:55:00,2023,11,24,Friday
15846,2023-11-24,4567.75,4568.75,4567.25,4568.25,2007.0,4567.900,529,09:57:00,09:57:00,2023,11,24,Friday
15847,2023-11-24,4568.25,4571.00,4568.00,4569.50,3984.0,4569.200,896,10:00:00,10:00:00,2023,11,24,Friday
15848,2023-11-24,4569.25,4569.75,4568.75,4569.25,1370.0,4569.275,444,10:04:00,10:04:00,2023,11,24,Friday


## 10.

In [37]:
import pandas as pd

barsCombined['date'] = pd.to_datetime(barsCombined['date']).dt.date

# CME early-close and holiday dates excluded from analysis
# (shortened sessions produce abnormal close prices that skew strategy results)
# Add new dates as needed — check: https://www.cmegroup.com/tools-information/holiday-calendar.html
EXCLUDED_DATES = [
    pd.to_datetime('2024-11-29').date(),  # Black Friday (early close)
    pd.to_datetime('2024-12-24').date(),  # Christmas Eve (early close)
    pd.to_datetime('2024-07-03').date(),  # Day before Independence Day (early close)
    pd.to_datetime('2023-07-03').date(),  # Day before Independence Day (early close)
    pd.to_datetime('2023-11-24').date(),  # Black Friday (early close)
]

barsCombined = barsCombined[~barsCombined['date'].isin(EXCLUDED_DATES)]

# Reset index so there are no gaps after filtering
barsCombined = barsCombined.reset_index(drop=True)

barsCombined.head(15850)


,date,candle_open_price,candle_high_price,candle_low_price,candle_close_price,volume,average,bar_count,time,confirm_time,year,month,day,day_of_week
0,2022-12-12,3976.25,3981.50,3974.75,3980.00,13850.0,3977.425,4176,09:30:00,09:30:00,2022,12,12,Monday
1,2022-12-12,3980.00,3980.50,3976.00,3977.50,8373.0,3977.900,2593,09:31:00,09:31:00,2022,12,12,Monday
2,2022-12-12,3977.25,3979.25,3975.25,3978.50,5649.0,3977.150,1895,09:32:00,09:32:00,2022,12,12,Monday
3,2022-12-12,3978.50,3979.75,3975.50,3978.75,5903.0,3978.200,2027,09:33:00,09:33:00,2022,12,12,Monday
4,2022-12-12,3978.50,3979.00,3973.75,3974.50,5902.0,3975.675,1996,09:34:00,09:34:00,2022,12,12,Monday
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15845,2023-11-28,4556.25,4556.25,4556.00,4556.25,141.0,4556.250,37,09:30:40,09:30:40,2023,11,28,Tuesday
15846,2023-11-28,4555.75,4555.75,4555.75,4555.75,46.0,4555.750,14,09:30:45,09:30:45,2023,11,28,Tuesday
15847,2023-11-28,4555.00,4555.25,4554.75,4555.00,199.0,4555.000,70,09:30:50,09:30:50,2023,11,28,Tuesday
15848,2023-11-28,4554.25,4554.50,4554.25,4554.25,50.0,4554.275,17,09:30:55,09:30:55,2023,11,28,Tuesday


## 11.

In [40]:
import pandas as pd

# Convert 'date' column to a datetime object
barsCombined["date"] = pd.to_datetime(barsCombined["date"])

barsCombined["date"] = barsCombined["date"].dt.strftime("%Y-%m-%d")

# Assign `candle_open_price` where time is `09:30:00`
barsCombined.loc[barsCombined["time"] == "09:30:00", "market_open_09:30"] = barsCombined["candle_open_price"]

# Forward-fill missing values so every row has the last seen 09:30:00 price
barsCombined["market_open_09:30"] = barsCombined["market_open_09:30"].ffill()

barsCombined.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 38142 entries, 0 to 38141
Data columns (total 15 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   date                38142 non-null  object 
 1   candle_open_price   38142 non-null  float64
 2   candle_high_price   38142 non-null  float64
 3   candle_low_price    38142 non-null  float64
 4   candle_close_price  38142 non-null  float64
 5   volume              38142 non-null  float64
 6   average             38142 non-null  float64
 7   bar_count           38142 non-null  int64  
 8   time                38142 non-null  object 
 9   confirm_time        38142 non-null  object 
 10  year                38142 non-null  int32  
 11  month               38142 non-null  int32  
 12  day                 38142 non-null  int32  
 13  day_of_week         38142 non-null  object 
 14  market_open_09:30   38142 non-null  float64
dtypes: float64(7), int32(3), int64(1), object(4)
memory u

In [42]:
barsCombined.head()

,date,candle_open_price,candle_high_price,candle_low_price,candle_close_price,volume,average,bar_count,time,confirm_time,year,month,day,day_of_week,market_open_09:30
0,2022-12-12,3976.25,3981.50,3974.75,3980.00,13850.0,3977.425,4176,09:30:00,09:30:00,2022,12,12,Monday,3976.25
1,2022-12-12,3980.00,3980.50,3976.00,3977.50,8373.0,3977.900,2593,09:31:00,09:31:00,2022,12,12,Monday,3976.25
2,2022-12-12,3977.25,3979.25,3975.25,3978.50,5649.0,3977.150,1895,09:32:00,09:32:00,2022,12,12,Monday,3976.25
3,2022-12-12,3978.50,3979.75,3975.50,3978.75,5903.0,3978.200,2027,09:33:00,09:33:00,2022,12,12,Monday,3976.25
4,2022-12-12,3978.50,3979.00,3973.75,3974.50,5902.0,3975.675,1996,09:34:00,09:34:00,2022,12,12,Monday,3976.25


## 12.

In [45]:
import pandas as pd
import numpy as np

# Vectorized initial direction — replaces a slow row-by-row .apply() function.
# 'above': bar opened higher than the market open -> expect reversal DOWN
# 'below': bar opened lower than the market open  -> expect reversal UP
# 'equal': bar opened exactly at market open (filtered out in next notebook)
barsCombined['initial_direction'] = np.where(
    barsCombined['candle_open_price'] > barsCombined['market_open_09:30'], 'above',
    np.where(barsCombined['candle_open_price'] < barsCombined['market_open_09:30'], 'below', 'equal')
)

barsCombined.to_csv('newBarsCombined.csv', index=False)


## 13.
## xlsx file is imported and contains contract locations above and below market open. Script then assigns either 'above' value or 'below' value to each row dependent on 'initial_direction'

In [48]:
import pandas as pd
import numpy as np

# Load the contract location reference data (above/below strikes per date)
file_path = "fixedConLocUpTo03-5-24.xlsx"
df_contracts = pd.read_excel(file_path, sheet_name='Sheet1')
df_contracts = df_contracts[['date', 'above', 'below']]

# Convert date columns for merging
barsCombined['date'] = pd.to_datetime(barsCombined['date'])
df_contracts['date'] = pd.to_datetime(df_contracts['date'])

# Warn if the dataset has dates beyond the contract location file coverage
max_contract_date = df_contracts['date'].max()
missing_dates = barsCombined[barsCombined['date'] > max_contract_date]['date'].dt.date.unique()
if len(missing_dates) > 0:
    print(f"WARNING: {len(missing_dates)} trading day(s) have no contract location data.")
    print(f"  Contract file covers through: {max_contract_date.date()}")
    print(f"  Missing dates (first 5): {missing_dates[:5]}")
    print("  Add these to fixedConLocUpTo03-5-24.xlsx to include them in analysis.")
else:
    print(f"Contract location data covers all dates through {max_contract_date.date()}")

# Merge contract location reference into main dataframe
cleaning = barsCombined.merge(df_contracts, on='date', how='left')

# Vectorized contract_location assignment (replaces slow .iterrows() loop)
# Strategy logic:
#   initial_direction == 'below' -> expect UP reversal -> contract is the ABOVE strike
#   initial_direction == 'above' -> expect DOWN reversal -> contract is the BELOW strike
#   initial_direction == 'equal' -> no position (NaN)
conditions = [
    cleaning['initial_direction'] == 'below',
    cleaning['initial_direction'] == 'above',
]
choices = [cleaning['above'], cleaning['below']]
cleaning['contract_location'] = np.select(conditions, choices, default=np.nan)

print(cleaning.head(300))
cleaning.to_csv('cleaning.csv', index=False)


          date  candle_open_price  candle_high_price  candle_low_price  \
0   2022-12-12            3976.25            3981.50           3974.75   
1   2022-12-12            3980.00            3980.50           3976.00   
2   2022-12-12            3977.25            3979.25           3975.25   
3   2022-12-12            3978.50            3979.75           3975.50   
4   2022-12-12            3978.50            3979.00           3973.75   
..         ...                ...                ...               ...   
295 2022-12-19            3859.00            3860.50           3858.50   
296 2022-12-19            3858.75            3859.00           3857.00   
297 2022-12-19            3855.50            3856.75           3854.75   
298 2022-12-19            3846.50            3847.75           3846.00   
299 2022-12-19            3847.75            3848.00           3847.00   

     candle_close_price   volume   average  bar_count      time confirm_time  \
0               3980.00  13850.